## Setup

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

if not os.getcwd().endswith("etl"):
    os.chdir("./etl")

import requests
import pandas as pd
import pyspark
from io import BytesIO
from dotenv import load_dotenv
from pyspark.sql import functions as F, Window

In [111]:
load_dotenv()

JDBC_PATH = os.getenv("JDBC_PATH")
_DB_URL = os.getenv("DB_URL")
_DB_NAME = os.getenv("DB_NAME")
_DB_USER = os.getenv("DB_USER")
_DB_PASSWORD = os.getenv("DB_PASSWORD")

if JDBC_PATH is None or _DB_URL is None or _DB_NAME is None or _DB_USER is None or _DB_PASSWORD is None:
    raise EnvironmentError("Set .env vars!")
if not os.path.exists(JDBC_PATH) or not os.path.isfile(JDBC_PATH):
    raise FileNotFoundError("JDBC driver not found!")

DB_URL = f"jdbc:mysql://{_DB_URL}/{_DB_NAME}"
DB_PROPERTIES = {"user": _DB_USER, "password": _DB_PASSWORD, "driver": "com.mysql.cj.jdbc.Driver"}

In [112]:
spark = pyspark.sql.SparkSession.builder.appName("ETL TCC").config("spark.jars", JDBC_PATH).getOrCreate()

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StructType, StructField, FloatType

geolocator = Nominatim(user_agent="tcc")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

schema = StructType([
    StructField("lat", FloatType()),
    StructField("long", FloatType())
])
@pandas_udf(schema)
def get_lat_long(row: pd.Series) -> pd.DataFrame:
    coords = []
    for i in row.index:
        loc = geocode(row[i])
        coords.append({"lat": loc.latitude, "long": loc.longitude})
    return pd.DataFrame.from_records(coords)


@pandas_udf(StringType())
def get_district(row: pd.Series) -> pd.Series:
    return row.apply(lambda x: "Pinheiros")

## Extract

### Base bueiros

In [114]:
BUEIROS_URL = "https://www.der.sp.gov.br/WebSite/Arquivos/DadosAbertos/AtivosRodoviarios/Bueiros/Bueiros.xlsx"
res = requests.get(BUEIROS_URL)
res.raise_for_status()

In [115]:
content_wrapper = BytesIO(res.content)
pandas_df = pd.read_excel(content_wrapper)
df_raw_bueiros = spark.createDataFrame(pandas_df)

### OpenMeteo

### Alagamentos CGE

### Mocks

In [116]:
pandas_df = pd.read_csv("./mocks/distritos.csv").reset_index()
df_mock_distritos = spark.createDataFrame(pandas_df)

## Transform

#### 1. Distrito

In [118]:
df_transf_distritos = df_mock_distritos\
    .withColumnRenamed("index", "id").withColumn("cidade", F.lit("São Paulo"))

#### 2. Coordenada

In [ ]:
df_raw_coords = df_transf_distritos
df_transf_coords = df_raw_coords.withColumnRenamed("id", "endereco_id")\
    .withColumn("coords", get_lat_long("bairro"))\
    .withColumn("id", F.row_number().over(Window.orderBy(F.lit(1))))#.drop("bairro", "cidade")

#### 3. Tipo

In [117]:
df_transf_tipo = df_raw_bueiros.select("Tipo").drop_duplicates().withColumnRenamed("Tipo", "nome").withColumn("id", F.monotonically_increasing_id())

#### 4. Bueiro

In [153]:
geolocator.geocode("SP 081 KM0.7")

Location(Rodovia Heitor Penteado, Nova Campinas, Campinas, São Paulo, Região Sudeste, 13092-340, Brasil, (-22.9050999, -47.036209, 0.0))

In [ ]:
df_transf_bueiro = df_raw_bueiros.withColumnRenamed("Extensão (m)", "extensao").withColumnRenamed("Dimensão (m)", "dimensao")\
    .withColumn("endereco", F.concat(F.col("Rodovia"), F.lit(" KM"), F.col("KM").cast("string"))).select("extensao", "dimensao", "endereco")
df_transf_bueiro = df_transf_bueiro.withColumn("distrito", get_district("endereco"))
df_transf_bueiro.show()

+--------+--------+------------------+
|extensao|dimensao|          endereco|
+--------+--------+------------------+
|    11.0|     0.4|      SP 081 KM0.7|
|    11.0|     0.6|     SP 081 KM0.88|
|    27.5|     0.6|     SP 081 KM1.21|
|    15.0|     0.4|      SP 081 KM5.6|
|    11.0|     0.8|      SP 081 KM8.0|
|    11.0|     0.8|      SP 081 KM8.1|
|    11.0|     0.6|      SP 081 KM8.2|
|    11.0|     0.6|      SP 081 KM8.3|
|    11.0|     0.6|      SP 081 KM8.4|
|    10.0|     1.0|    SP 091 KM90.53|
|    15.0|     1.0|    SP 091 KM90.67|
|    20.0|     1.5| SPA 110/330 KM0.5|
|   18.23|     1.0|SPA 110/330 KM1.25|
|   20.14|     1.5|SPA 110/330 KM1.32|
|    10.0|     0.6|SPA 110/330 KM1.37|
|   41.93|     0.6|SPA 110/330 KM2.42|
|    36.7|     0.6|SPA 110/330 KM2.56|
|    6.48|     0.6|SPA 110/330 KM2.58|
|    11.0|     0.4|SPA 115/330 KM0.49|
|   19.29|     1.0|SPA 115/330 KM1.05|
+--------+--------+------------------+
only showing top 20 rows


Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/usr/local/lib/python3.10/dist-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


: 

## Load

### 1. Distrito

In [ ]:
df_transf_distritos.write.jdbc(DB_URL, "distrito", properties=DB_PROPERTIES, mode="append")

### 2. Coordenada

In [148]:
df_transf_coords.write.jdbc(DB_URL, "coordenada", properties=DB_PROPERTIES, mode="append")

26/04/21 18:26:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 18:26:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 18:26:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 18:26:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/21 18:26:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


### 3. Tipo

In [104]:
df_transf_tipo.write.jdbc(DB_URL, "tipo", properties=DB_PROPERTIES, mode="append")